# Chapter 2 Practical 03: User Profile and Ranking

Learning objectives:
- Build a user profile from liked movies.
- Compare average, rating-weighted, normalized, implicit-feedback, and temporal-decay profiles.
- Recommend unseen movies.
- Explain recommendations using overlapping features.

Slide connection: user profiles, similarity matching, ranking, and Top-N recommendation.


Load movie metadata and a small set of user interactions.


In [1]:
# Teaching note: Load movie metadata and interactions with a Colab-safe fallback. Local files are used first; GitHub raw CSVs are used when opened directly from GitHub.
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_02_content_based/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_02_content_based/data"

def read_chapter2_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

movies = read_chapter2_csv("movies_chapter2.csv")
interactions = read_chapter2_csv("user_interactions_chapter2.csv")

movies.head(), interactions.head()


Loaded movies_chapter2.csv from data/movies_chapter2.csv
Loaded user_interactions_chapter2.csv from data/user_interactions_chapter2.csv


(   movie_id         title                             genres  \
 0         1     Inception             Sci-Fi|Thriller|Action   
 1         2  Interstellar             Sci-Fi|Adventure|Drama   
 2         3       Titanic                      Romance|Drama   
 3         4    The Matrix                      Sci-Fi|Action   
 4         5     Toy Story  Animation|Adventure|Comedy|Family   
 
             director  year  duration_min  rating  family_friendly  \
 0  Christopher Nolan  2010           148     8.8                0   
 1  Christopher Nolan  2014           169     8.7                0   
 2      James Cameron  1997           195     7.9                0   
 3     The Wachowskis  1999           136     8.7                0   
 4      John Lasseter  1995            81     8.3                1   
 
                                          description  \
 0  A thief enters layered dreams to plant an idea...   
 1  Astronauts travel through a wormhole to find a...   
 2  A young cou

Create an item-feature matrix from genres, directors, and text. This gives the user profile a mix of structured and textual evidence.


In [2]:
# Teaching note: Create content features that combine genres, director names, and text metadata.
# TF-IDF converts text into weighted numeric features.
from sklearn.feature_extraction.text import TfidfVectorizer
# MinMaxScaler puts numeric features on a shared 0-1 scale.
from sklearn.preprocessing import MinMaxScaler
# Cosine similarity turns vectors into pairwise recommendation scores.
from sklearn.metrics.pairwise import cosine_similarity

movies["feature_text"] = (
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["director"] + " " +
    movies["description"] + " " +
    movies["keywords"]
)
# TF-IDF converts text into weighted numeric features.
tfidf = TfidfVectorizer(stop_words="english")
text_features = tfidf.fit_transform(movies["feature_text"]).toarray()

# MinMaxScaler puts numeric features on a shared 0-1 scale.
numeric = MinMaxScaler().fit_transform(movies[["duration_min", "rating", "family_friendly"]])
feature_names = list(tfidf.get_feature_names_out()) + ["duration_scaled", "rating_scaled", "family_friendly"]
item_features = pd.DataFrame(
    np.hstack([text_features, numeric]),
    index=movies["title"],
    columns=feature_names,
)
item_features.iloc[:5, -8:].round(2)


,uses,wachowskis,world,wormhole,young,duration_scaled,rating_scaled,family_friendly
title,,,,,,,,
Inception,0.0,0.00,0.00,0.00,0.00,0.59,0.88,0.0
Interstellar,0.0,0.00,0.00,0.55,0.00,0.77,0.82,0.0
Titanic,0.0,0.00,0.00,0.00,0.21,1.00,0.35,0.0
The Matrix,0.0,0.23,0.23,0.00,0.00,0.48,0.82,0.0
Toy Story,0.0,0.00,0.00,0.00,0.00,0.00,0.59,1.0


For a simple average profile, each liked item has the same influence.


In [3]:
# Teaching note: Build a simple average profile from movies the user liked.
user_id = "U1"
user_history = interactions[interactions["user_id"] == user_id].copy()
liked_titles = user_history["title"].tolist()

average_profile = item_features.loc[liked_titles].mean(axis=0)
average_profile.sort_values(ascending=False).head(10).round(3)


rating_scaled      0.843
duration_scaled    0.614
wormhole           0.183
action             0.177
sci                0.153
fi                 0.153
reality            0.153
hacker             0.153
dreams             0.138
christopher        0.130
dtype: float64

A rating-weighted profile gives stronger liked items more influence.


In [4]:
# Teaching note: Build a rating-weighted profile so stronger feedback has more influence.
weights_rating = user_history.set_index("title")["rating"]
rating_weighted_profile = item_features.loc[liked_titles].mul(weights_rating, axis=0).sum() / weights_rating.sum()
rating_weighted_profile.sort_values(ascending=False).head(10).round(3)


rating_scaled      0.845
duration_scaled    0.623
wormhole           0.196
action             0.165
sci                0.154
fi                 0.154
dreams             0.148
christopher        0.140
nolan              0.140
hacker             0.131
dtype: float64

A rating-normalized profile centers ratings around the user's average. This reduces the effect of users who rate everything high.


In [5]:
# Teaching note: Normalize feedback weights to compare users with different rating habits.
normalized_weights = weights_rating - weights_rating.mean()
if normalized_weights.abs().sum() == 0:
    normalized_weights = weights_rating / weights_rating.sum()

rating_normalized_profile = item_features.loc[liked_titles].mul(normalized_weights, axis=0).sum()
rating_normalized_profile.sort_values(ascending=False).head(10).round(3)


wormhole           0.183
dreams             0.138
duration_scaled    0.132
nolan              0.130
christopher        0.130
humanity           0.091
travel             0.091
exploration        0.091
mind               0.080
questioning        0.080
dtype: float64

Implicit feedback can combine clicks, likes, and watch time. Temporal decay gives recent interactions more weight.


In [6]:
# Teaching note: Use implicit signals such as clicks, likes, and watch time as profile weights.
max_duration = movies.set_index("title")["duration_min"]
history = user_history.set_index("title")
watch_ratio = history["watch_minutes"] / max_duration.loc[history.index]
implicit_weight = 0.2 * history["clicked"] + 0.5 * history["liked"] + 0.3 * watch_ratio
# Exponential decay gives recent interactions larger weights than old ones.
temporal_decay = np.exp(-history["days_ago"] / 30)
final_weight = implicit_weight * temporal_decay

profile_temporal = item_features.loc[history.index].mul(final_weight, axis=0).sum() / final_weight.sum()
pd.DataFrame({
    "implicit_weight": implicit_weight.round(3),
    "temporal_decay": temporal_decay.round(3),
    "final_weight": final_weight.round(3),
})


,implicit_weight,temporal_decay,final_weight
title,,,
Inception,0.984,0.905,0.890
Interstellar,0.993,0.670,0.666
The Matrix,0.965,0.368,0.355


Recommend unseen movies by comparing each item vector to the user profile.


In [7]:
# Teaching note: Apply temporal decay so recent interactions influence the profile more.
def recommend_from_profile(profile, seen_titles, top_n=5):
    candidate_features = item_features.drop(index=seen_titles)
    # Cosine similarity turns vectors into pairwise recommendation scores.
    scores = cosine_similarity(candidate_features, profile.values.reshape(1, -1)).ravel()
    results = pd.DataFrame({"title": candidate_features.index, "score": scores})
    return results.sort_values("score", ascending=False).head(top_n)

recommendations = recommend_from_profile(profile_temporal, liked_titles)
recommendations.round(3)


,title,score
3,The Dark Knight,0.694
0,Titanic,0.516
4,The Martian,0.509
8,La La Land,0.454
5,The Notebook,0.360


We can explain each recommendation by showing the strongest features shared by the user profile and the movie.


In [8]:
# Teaching note: Rank unseen movies and attach feature-overlap explanations.
def explain_recommendation(title, profile, top_n=6):
    contribution = item_features.loc[title] * profile
    return contribution.sort_values(ascending=False).head(top_n).round(3)

best_title = recommendations.iloc[0]["title"]
print(f"Explanation for {best_title}:")
explain_recommendation(best_title, profile_temporal)


Explanation for The Dark Knight:


rating_scaled      0.851
duration_scaled    0.394
nolan              0.030
christopher        0.030
action             0.028
drama              0.008
dtype: float64

## What did we learn?

- A user profile is a vector summarizing what the user liked.
- Different weighting choices create different profiles.
- Recommendations become more transparent when we inspect shared high-weight features.

## Challenge Lab

1. Change `user_id` and compare the average, rating-weighted, implicit-feedback, and temporal-decay profiles.
2. Increase or decrease the temporal decay strength. Which old interaction loses influence first?
3. Add one disliked movie with a negative weight and update the profile formula. How do the recommendations change?
4. Write a one-sentence explanation for the top recommendation using the strongest profile features.
